Read Bronze Subscriptions

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.table(
    "customer360_dev.bronze.subscriptions"
)

customers_df = (
    spark.table("customer360_dev.silver.customers")
    .select("customer_id")
)

Cast and standardize fields

In [0]:
typed_df = (
    bronze_df
    .withColumn(
        "start_date_cast",
        F.to_date("start_date", "yyyy-MM-dd")
    )
    .withColumn(
        "end_date_cast",
        F.to_date("end_date", "yyyy-MM-dd")
    )
    .withColumn(
        "monthly_fee_cast",
        F.col("monthly_fee").cast("decimal(10,2)")
    )
    .withColumn(
        "auto_renew_cast",
        F.col("auto_renew").cast("boolean")
    )
    .withColumn(
        "trial_flag_cast",
        F.col("trial_flag").cast("boolean")
    )
    .withColumn(
        "created_at_cast",
        F.to_timestamp("created_at")
    )
    .withColumn(
        "updated_at_cast",
        F.to_timestamp("updated_at")
    )
    .withColumn(
        "plan_code_clean",
        F.upper(F.trim("plan_code"))
    )
    .withColumn(
        "subscription_status_clean",
        F.upper(F.trim("subscription_status"))
    )
    .withColumn(
        "billing_cycle_clean",
        F.upper(F.trim("billing_cycle"))
    )
    .withColumn(
        "payment_method_type_clean",
        F.upper(F.trim("payment_method_type"))
    )
)

Add customer referential check

In [0]:
validated_df = (
    typed_df
    .join(
        customers_df.withColumn(
            "_customer_exists",
            F.lit(True)
        ),
        on="customer_id",
        how="left"
    )
)

Define data-quality rules

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "dq_missing_subscription_id",
        F.col("subscription_id").isNull()
        | (F.trim(F.col("subscription_id")) == "")
    )
    .withColumn(
        "dq_missing_customer_id",
        F.col("customer_id").isNull()
        | (F.trim(F.col("customer_id")) == "")
    )
    .withColumn(
        "dq_customer_not_found",
        F.col("_customer_exists").isNull()
    )
    .withColumn(
        "dq_invalid_status",
        ~F.col("subscription_status_clean").isin(
            "ACTIVE",
            "PAUSED",
            "CANCELLED",
            "EXPIRED"
        )
    )
    .withColumn(
        "dq_invalid_plan",
        ~F.col("plan_code_clean").isin(
            "BASIC",
            "STANDARD",
            "PREMIUM",
            "ENTERPRISE"
        )
    )
    .withColumn(
        "dq_invalid_billing_cycle",
        ~F.col("billing_cycle_clean").isin(
            "MONTHLY",
            "QUARTERLY",
            "YEARLY"
        )
    )
    .withColumn(
        "dq_negative_fee",
        F.col("monthly_fee_cast").isNull()
        | (F.col("monthly_fee_cast") < 0)
    )
    .withColumn(
        "dq_invalid_date_range",
        F.col("start_date_cast").isNull()
        | (
            F.col("end_date_cast").isNotNull()
            & (F.col("end_date_cast") < F.col("start_date_cast"))
        )
    )
)

Add lifecycle consistency checks

In [0]:
validated_df = validated_df.withColumn(
    "dq_warning_cancelled_without_end_date",
    (
        (F.col("subscription_status_clean") == "CANCELLED")
        & F.col("end_date_cast").isNull()
    )
)

validated_df = validated_df.withColumn(
    "dq_warning_expired_without_end_date",
    (
        (F.col("subscription_status_clean") == "EXPIRED")
        & F.col("end_date_cast").isNull()
    )
)

In [0]:
validated_df.select("dq_invalid_billing_cycle").distinct().display()

dq_invalid_billing_cycle
false
null


In [0]:
display(validated_df.
    filter(F.col("dq_invalid_billing_cycle") |
           F.col("dq_invalid_date_range")).limit(10))

customer_id,subscription_id,plan_code,subscription_status,start_date,end_date,billing_cycle,monthly_fee,auto_renew,payment_method_type,trial_flag,cancel_reason,created_at,updated_at,_ingestion_timestamp,_source_file,_source_system,_load_type,start_date_cast,end_date_cast,monthly_fee_cast,auto_renew_cast,trial_flag_cast,created_at_cast,updated_at_cast,plan_code_clean,subscription_status_clean,billing_cycle_clean,payment_method_type_clean,_customer_exists,dq_missing_subscription_id,dq_missing_customer_id,dq_customer_not_found,dq_invalid_status,dq_invalid_plan,dq_invalid_billing_cycle,dq_negative_fee,dq_invalid_date_range,dq_warning_cancelled_without_end_date,dq_warning_expired_without_end_date
CUST000000533,SUB0000000041,ENTERPRISE,EXPIRED,2025-03-27,2025-03-17,QUARTERLY,2499.00,true,WALLET,false,null,2024-02-11 23:31:53,2024-03-01 12:00:00,2026-09-26T09:59:34.935Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/initial/subscriptions_initial_001.csv,SUBSCRIPTION_PLATFORM,INITIAL,2025-03-27,2025-03-17,2499.00,true,false,2024-02-11T23:31:53.000Z,2024-03-01T12:00:00.000Z,ENTERPRISE,EXPIRED,QUARTERLY,WALLET,true,false,false,false,false,false,false,false,true,false,false
CUST000007033,SUB0000000541,ENTERPRISE,EXPIRED,2026-08-19,2026-08-09,YEARLY,2499.00,true,WALLET,false,null,2025-06-25 19:11:13,2025-07-14 08:40:20,2026-09-26T09:59:34.935Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/initial/subscriptions_initial_001.csv,SUBSCRIPTION_PLATFORM,INITIAL,2026-08-19,2026-08-09,2499.00,true,false,2025-06-25T19:11:13.000Z,2025-07-14T08:40:20.000Z,ENTERPRISE,EXPIRED,YEARLY,WALLET,true,false,false,false,false,false,false,false,true,false,false
CUST000013533,SUB0000001041,ENTERPRISE,EXPIRED,2025-04-18,2025-04-08,MONTHLY,2499.00,false,WALLET,false,null,2024-02-12 15:51:33,2024-03-02 04:20:40,2026-09-26T09:59:34.935Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/initial/subscriptions_initial_001.csv,SUBSCRIPTION_PLATFORM,INITIAL,2025-04-18,2025-04-08,2499.00,false,false,2024-02-12T15:51:33.000Z,2024-03-02T04:20:40.000Z,ENTERPRISE,EXPIRED,MONTHLY,WALLET,true,false,false,false,false,false,false,false,true,false,false
CUST000020033,SUB0000001541,ENTERPRISE,EXPIRED,2026-09-10,2026-08-31,QUARTERLY,2499.00,true,WALLET,false,null,2025-06-26 11:31:53,2025-07-15 00:00:00,2026-09-26T09:59:34.935Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/initial/subscriptions_initial_001.csv,SUBSCRIPTION_PLATFORM,INITIAL,2026-09-10,2026-08-31,2499.00,true,false,2025-06-26T11:31:53.000Z,2025-07-15T00:00:00.000Z,ENTERPRISE,EXPIRED,QUARTERLY,WALLET,true,false,false,false,false,false,false,false,true,false,false
CUST000026533,SUB0000002041,ENTERPRISE,EXPIRED,2025-05-10,2025-04-30,YEARLY,2499.00,true,WALLET,false,null,2024-02-13 07:11:13,2024-03-03 20:40:20,2026-09-26T09:59:34.935Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/initial/subscriptions_initial_001.csv,SUBSCRIPTION_PLATFORM,INITIAL,2025-05-10,2025-04-30,2499.00,true,false,2024-02-13T07:11:13.000Z,2024-03-03T20:40:20.000Z,ENTERPRISE,EXPIRED,YEARLY,WALLET,true,false,false,false,false,false,false,false,true,false,false
CUST000033033,SUB0000002541,ENTERPRISE,EXPIRED,2024-01-08,2023-12-29,MONTHLY,2499.00,false,WALLET,false,null,2025-06-27 03:51:33,2025-07-16 16:20:40,2026-09-26T09:59:34.935Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/initial/subscriptions_initial_001.csv,SUBSCRIPTION_PLATFORM,INITIAL,2024-01-08,2023-12-29,2499.00,false,false,2025-06-27T03:51:33.000Z,2025-07-16T16:20:40.000Z,ENTERPRISE,EXPIRED,MONTHLY,WALLET,true,false,false,false,false,false,false,false,true,false,false
CUST000039533,SUB0000003041,ENTERPRISE,EXPIRED,2025-06-01,2025-05-22,QUARTERLY,2499.00,true,WALLET,false,null,2024-02-14 23:31:53,2024-03-04 12:00:00,2026-09-26T09:59:34.935Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/initial/subscriptions_initial_001.csv,SUBSCRIPTION_PLATFORM,INITIAL,2025-06-01,20

Define critical failure logic

In [0]:
validated_df = validated_df.withColumn(
    "dq_critical_failure",
    F.col("dq_missing_subscription_id")
    | F.col("dq_missing_customer_id")
    | F.col("dq_customer_not_found")
    | F.col("dq_invalid_status")
    | F.col("dq_invalid_plan")
    | F.col("dq_invalid_billing_cycle")
    | F.col("dq_negative_fee")
    | F.col("dq_invalid_date_range")
)

In [0]:
validated_df.select("dq_critical_failure").distinct().display()

dq_critical_failure
false
true


Split valid and rejected rows

In [0]:
reject_df = validated_df.filter(
    F.col("dq_critical_failure")
)

valid_df = validated_df.filter(
    ~F.col("dq_critical_failure")
)

In [0]:
display(valid_df.limit(5))

customer_id,subscription_id,plan_code,subscription_status,start_date,end_date,billing_cycle,monthly_fee,auto_renew,payment_method_type,trial_flag,cancel_reason,created_at,updated_at,_ingestion_timestamp,_source_file,_source_system,_load_type,start_date_cast,end_date_cast,monthly_fee_cast,auto_renew_cast,trial_flag_cast,created_at_cast,updated_at_cast,plan_code_clean,subscription_status_clean,billing_cycle_clean,payment_method_type_clean,_customer_exists,dq_missing_subscription_id,dq_missing_customer_id,dq_customer_not_found,dq_invalid_status,dq_invalid_plan,dq_invalid_billing_cycle,dq_negative_fee,dq_invalid_date_range,dq_warning_cancelled_without_end_date,dq_warning_expired_without_end_date,dq_critical_failure
CUST000000013,SUB0000000001,ENTERPRISE,EXPIRED,2024-01-12,2024-04-12,YEARLY,2499.00,true,WALLET,false,null,2024-01-02 07:11:13,2024-01-21 20:40:20,2026-09-26T09:59:34.935Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/initial/subscriptions_initial_001.csv,SUBSCRIPTION_PLATFORM,INITIAL,2024-01-12,2024-04-12,2499.00,true,false,2024-01-02T07:11:13.000Z,2024-01-21T20:40:20.000Z,ENTERPRISE,EXPIRED,YEARLY,WALLET,true,false,false,false,false,false,false,false,false,false,false,false
CUST000000026,SUB0000000002,PREMIUM,CANCELLED,2024-01-23,2024-04-24,QUARTERLY,999.00,true,NETBANKING,false,null,2024-01-03 14:22:26,2024-01-22 03:51:33,2026-09-26T09:59:34.935Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/initial/subscriptions_initial_001.csv,SUBSCRIPTION_PLATFORM,INITIAL,2024-01-23,2024-04-24,999.00,true,false,2024-01-03T14:22:26.000Z,2024-01-22T03:51:33.000Z,PREMIUM,CANCELLED,QUARTERLY,NETBANKING,true,false,false,false,false,false,false,false,false,false,false,false
CUST000000039,SUB0000000003,STANDARD,PAUSED,2024-02-03,null,MONTHLY,499.00,false,UPI,false,null,2024-01-04 21:33:39,2024-01-23 10:02:46,2026-09-26T09:59:34.935Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/initial/subscriptions_initial_001.csv,SUBSCRIPTION_PLATFORM,INITIAL,2024-02-03,null,499.00,false,false,2024-01-04T21:33:39.000Z,2024-01-23T10:02:46.000Z,STANDARD,PAUSED,MONTHLY,UPI,true,false,false,false,false,false,false,false,false,false,false,false
CUST000000052,SUB0000000004,BASIC,ACTIVE,2024-02-14,null,YEARLY,199.00,true,CARD,false,null,2024-01-05 04:44:52,2024-01-24 17:13:59,2026-09-26T09:59:34.935Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/initial/subscriptions_initial_001.csv,SUBSCRIPTION_PLATFORM,INITIAL,2024-02-14,null,199.00,true,false,2024-01-05T04:44:52.000Z,2024-01-24T17:13:59.000Z,BASIC,ACTIVE,YEARLY,CARD,true,false,false,false,false,false,false,false,false,false,false,false
CUST000000065,SUB0000000005,ENTERPRISE,EXPIRED,2024-02-25,2024-05-30,QUARTERLY,2499.00,true,WALLET,false,null,2024-01-06 11:55:05,2024-01-25 00:24:12,2026-09-26T09:59:34.935Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/initial/subscriptions_initial_001.csv,SUBSCRIPTION_PLATFORM,INITIAL,2024-02-25,2024-05-30,2499.00,true,false,2024-01-06T11:55:05.000Z,2024-01-25T00:24:12.000Z,ENTERPRISE,EXPIRED,QUARTERLY,WALLET,true,false,false,false,false,false,false,false,false,false,false,false


Deduplicate deterministically

In [0]:
subscription_window = (
    Window
    .partitionBy("subscription_id")
    .orderBy(
        F.col("updated_at_cast").desc_nulls_last(),
        F.col("_ingestion_timestamp").desc()
    )
)

deduped_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(subscription_window)
    )
    .filter(F.col("_row_number") == 1)
)

Build final Silver structure

In [0]:
silver_df = deduped_df.select(
    "subscription_id",
    "customer_id",

    F.col("plan_code_clean").alias("plan_code"),
    F.col("subscription_status_clean").alias(
        "subscription_status"
    ),

    F.col("start_date_cast").alias("start_date"),
    F.col("end_date_cast").alias("end_date"),

    F.col("billing_cycle_clean").alias("billing_cycle"),
    F.col("monthly_fee_cast").alias("monthly_fee"),
    F.col("auto_renew_cast").alias("auto_renew"),

    F.col("payment_method_type_clean").alias(
        "payment_method_type"
    ),

    F.col("trial_flag_cast").alias("trial_flag"),
    "cancel_reason",

    F.col("created_at_cast").alias("created_at"),
    F.col("updated_at_cast").alias("updated_at"),

    "dq_warning_cancelled_without_end_date",
    "dq_warning_expired_without_end_date",

    "_source_file",
    "_source_system",
    "_load_type",
    "_ingestion_timestamp",

    F.current_timestamp().alias(
        "_silver_processed_at"
    )
)

Write Silver Subscriptions

In [0]:
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "customer360_dev.silver.subscriptions"
    )
)

Build Quarantine records

In [0]:
quarantine_df = (
    reject_df
    .withColumn(
        "rejection_reason",
        F.concat_ws(
            "; ",
            F.when(
                F.col("dq_missing_subscription_id"),
                F.lit("MISSING_SUBSCRIPTION_ID")
            ),
            F.when(
                F.col("dq_missing_customer_id"),
                F.lit("MISSING_CUSTOMER_ID")
            ),
            F.when(
                F.col("dq_customer_not_found"),
                F.lit("CUSTOMER_NOT_FOUND")
            ),
            F.when(
                F.col("dq_invalid_status"),
                F.lit("INVALID_SUBSCRIPTION_STATUS")
            ),
            F.when(
                F.col("dq_invalid_plan"),
                F.lit("INVALID_PLAN")
            ),
            F.when(
                F.col("dq_invalid_billing_cycle"),
                F.lit("INVALID_BILLING_CYCLE")
            ),
            F.when(
                F.col("dq_negative_fee"),
                F.lit("INVALID_MONTHLY_FEE")
            ),
            F.when(
                F.col("dq_invalid_date_range"),
                F.lit("INVALID_DATE_RANGE")
            )
        )
    )
    .withColumn(
        "_quarantined_at",
        F.current_timestamp()
    )
)

In [0]:
if "_pipeline_run_id" not in spark.table("customer360_dev.quarantine.subscription_rejects").columns:
    spark.sql("ALTER TABLE customer360_dev.quarantine.subscription_rejects ADD COLUMNS (_pipeline_run_id STRING)")
(
    quarantine_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "customer360_dev.quarantine.subscription_rejects"
    )
)

Verify counts

In [0]:
bronze_count = spark.table(
    "customer360_dev.bronze.subscriptions"
).count()

silver_count = spark.table(
    "customer360_dev.silver.subscriptions"
).count()

reject_count = spark.table(
    "customer360_dev.quarantine.subscription_rejects"
).count()

print(f"Bronze : {bronze_count:,}")
print(f"Silver : {silver_count:,}")
print(f"Reject : {reject_count:,}")

Bronze : 1,000,066
Silver : 978,054
Reject : 22,012
